### Model Context Protocol

In [17]:
from fastmcp import FastMCP
import json

In [4]:
mcp=FastMCP(
    "First Server"
)

In [ ]:
# Tool
@mcp.tool
def add(a:int,b:int) -> int:
    """This recieves two numbers and return their sum"""
    return a+b

### Connect with FastMCP client

In [7]:
from fastmcp import Client

In [12]:
async with Client(mcp) as client:
    print("Protocol Version",client.protocol_version)
    tools=await client.list_tools()
    print("Tools",tools)

    print("Tool Calling")
    result=await client.call_tool("add",{'a':5,'b':3})
    print(result.data)

Protocol Version 2026-07-28
Tools [Tool(name='add', title='Add', description='This recieves two numbers and return their sum', input_schema={'type': 'object', 'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}, execution=None, output_schema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}})]
Tool Calling
8


C:\Users\HP\AppData\Local\Temp\ipykernel_16444\3380716553.py:7: RuntimeWarning: coroutine 'ClientToolsMixin.call_tool' was never awaited
  result=await client.call_tool("add",{'a':5,'b':3})


### MCP Tools -> Context, Tasks , Auth

In [18]:
async def show(server:FastMCP,tool_name:str):
    async with Client(server) as client:
        tool=None
        for t in await client.list_tools():
            if t.name==tool_name:
                tool=t
                break
        print(f"Tools:{tool.name}:{tool.description}")
        print(json.dumps(tool.input_schema,indent=2))

In [19]:
await show(mcp,'add')

Tools:add:This recieves two numbers and return their sum
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}


In [22]:
@mcp.tool
def multiply(a:int, b:int=1) -> int:
    """Takes two arguments and return their multiplication"""
    return a*b

[11/08/26 14:06:18] WARNING  Component already exists:    local_provider.py:192
                             tool:multiply@                                    


In [23]:
await show(mcp,'multiply')

Tools:multiply:Takes two arguments and return their multiplication
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "default": 1,
      "type": "integer"
    }
  },
  "required": [
    "a"
  ]
}


In [24]:
from typing import Literal

In [26]:
@mcp.tool
def export(table:str,format:Literal['csv','json','excel']='csv'):
    """Return the table in the required format"""
    return 'Export Done'

In [29]:
await show(mcp,'export')

Tools:export:Return the table in the required format
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "table": {
      "type": "string"
    },
    "format": {
      "default": "csv",
      "enum": [
        "csv",
        "json",
        "excel"
      ],
      "type": "string"
    }
  },
  "required": [
    "table"
  ]
}


### Pydantic model as a Argument

In [36]:
from pydantic import BaseModel , Field

class Shipment(BaseModel):
    order_id:str=Field(description="This is the order id placed by the user")
    quantity:int=Field(description="This is the quantity")
    express:bool=False

@mcp.tool
def book_shipment(shipment:Shipment):
    """Book a shipment for a order"""
    return {"Booked":shipment.order_id,"qty":shipment.quantity}

[11/08/26 14:16:53] WARNING  Component already exists:    local_provider.py:192
                             tool:book_shipment@                               


In [37]:
await show(mcp,'book_shipment')

Tools:book_shipment:Book a shipment for a order
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "shipment": {
      "properties": {
        "order_id": {
          "description": "This is the order id placed by the user",
          "type": "string"
        },
        "quantity": {
          "description": "This is the quantity",
          "type": "integer"
        },
        "express": {
          "default": false,
          "type": "boolean"
        }
      },
      "required": [
        "order_id",
        "quantity"
      ],
      "type": "object"
    }
  },
  "required": [
    "shipment"
  ]
}


### Exception/Error Handling